# FOMC-Dated Swap Analytics (FOMCAnalyzer)

Uses `SDRUtils.analytics.fomc.FOMCAnalyzer` — dual-curve (SOFR/OIS) analytics with:
- Dynamic SOFR/EFFR fixings from NY Fed API
- BARCHART_STIRF-RL short-end curves
- Accrued fixing stripping for in-progress meetings
- SDR trade flow decomposition by meeting
- Trade quality flagging (UFRO detection)

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', '..'))

import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import _sdr_common as sdr
sdr.notebook_setup()

from SDRUtils.analytics.fomc import (
    FOMCAnalyzer,
    classify_rate_index,
    compute_calendar_spreads,
    compute_cut_probabilities,
)
from SDRUtils.analytics.trade_quality import flag_outliers, TradeQualityFlag
from SDRUtils.analytics.intraday import hourly_distribution, trade_clustering
from SDRUtils.analytics.flow import classify_venue

# --- Parameters ---
START = datetime.datetime(2026, 1, 10, tzinfo=datetime.timezone.utc)
END = datetime.datetime(2026, 4, 10, 23, 59, 59, tzinfo=datetime.timezone.utc)
CACHE_PATH = sdr.DEFAULT_CACHE_PATH

In [ ]:
# Load classified trades
df = sdr.load_classified_trades(START, END, cache_path=CACHE_PATH)
print(f"Loaded {len(df):,} trades from {df['execution_date'].nunique()} trading days")
print(f"Date range: {df['execution_date'].min()} to {df['execution_date'].max()}")

## 1. Initialize FOMCAnalyzer

Single entry point for all FOMC analytics. Fetches fixings dynamically,
builds both SOFR and OIS curves, loads meeting schedule.

In [ ]:
analyzer = FOMCAnalyzer(
    df,
    sofr_curve_name="USD-SOFR-1D-Q12xM12STIRT",
    ois_curve_name="USD-OIS-Q12xM12STIRT-SERFFX-MIX23",
    # current_sofr=None → fetched dynamically from NY Fed
    # current_effr=None → fetched dynamically
)

print(f"Current SOFR: {analyzer.current_sofr*100:.4f}%")
print(f"Current EFFR: {analyzer.current_effr*100:.4f}%")
print(f"SOFR-EFFR spread: {(analyzer.current_sofr - analyzer.current_effr)*10000:.1f}bp")
print(f"FOMC meetings in schedule: {len(analyzer.schedule)}")
print(f"FOMC-dated trades: {len(analyzer.fomc_trades):,}")

## 2. FOMC Meeting Schedule

In [ ]:
display(analyzer.schedule.head(15))

## 3. Compute Implied Rates (Dual-Curve)

`.compute()` runs the full pipeline:
1. Build SOFR + OIS curves (BARCHART_STIRF-RL)
2. Price each meeting on both curves
3. Strip accrued fixings from in-progress meetings
4. Merge SDR VWAP rates
5. Compute implied moves, SOFR-OIS basis, cut probabilities

In [ ]:
implied_df = analyzer.compute()

# Display key columns
display_cols = ["meeting_label", "effective_date", "period_days",
                "sofr_implied_rate", "ois_implied_rate",
                "sofr_fwd_rate", "ois_fwd_rate",
                "accrued_days", "remaining_days"]
display_df = implied_df[display_cols].copy()
for col in ["sofr_implied_rate", "ois_implied_rate", "sofr_fwd_rate", "ois_fwd_rate"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x*100:.3f}%" if pd.notna(x) else "N/A")
display(display_df)

## 4. FOMC Curve Plot — SOFR vs OIS

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

meeting_dates = implied_df["effective_date"]
sofr = implied_df["sofr_fwd_rate"] * 100
ois = implied_df["ois_fwd_rate"] * 100

ax.plot(meeting_dates, sofr, marker="o", color="#4C78A8", linewidth=2,
        markersize=8, label="SOFR (forward)", zorder=3)
ax.plot(meeting_dates, ois, marker="s", color="#F58518", linewidth=2,
        markersize=7, label="OIS/FF (forward)", zorder=3)

ax.axhline(analyzer.current_sofr * 100, color="#4C78A8", linestyle="--",
           alpha=0.4, label=f"SOFR fixing ({analyzer.current_sofr*100:.2f}%)")
ax.axhline(analyzer.current_effr * 100, color="#F58518", linestyle="--",
           alpha=0.4, label=f"EFFR fixing ({analyzer.current_effr*100:.2f}%)")

for _, row in implied_df.iterrows():
    if pd.notna(row["sofr_fwd_rate"]):
        ax.annotate(row["meeting_label"],
                    (row["effective_date"], row["sofr_fwd_rate"] * 100),
                    textcoords="offset points", xytext=(0, 12),
                    fontsize=8, ha="center", color="#333")

ax.set_title("FOMC Curve: SOFR vs OIS Forward-Only Implied Rates")
ax.set_ylabel("Implied Rate (%)")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Cut Probabilities — Dual-Curve

In [ ]:
probs = compute_cut_probabilities(implied_df, analyzer.current_sofr, analyzer.current_effr)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

x = np.arange(len(probs))
w = 0.35
axes[0].bar(x - w/2, probs["p_cut_sofr"].fillna(0), w, color="#4C78A8", alpha=0.8, label="SOFR")
axes[0].bar(x + w/2, probs["p_cut_ois"].fillna(0), w, color="#F58518", alpha=0.8, label="OIS")
axes[0].axhline(50, color="black", linestyle="--", alpha=0.3)
axes[0].axhline(100, color="red", linestyle="--", alpha=0.3)
axes[0].set_xticks(x)
axes[0].set_xticklabels(probs["meeting_label"], rotation=45, ha="right")
axes[0].set_ylabel("P(25bp cut) %")
axes[0].set_title("Implied Probability of 25bp Cut: SOFR vs OIS")
axes[0].legend()

axes[1].plot(x, probs["cumulative_cuts_sofr"], marker="o", color="#4C78A8",
             linewidth=2, label="SOFR")
axes[1].plot(x, probs["cumulative_cuts_ois"], marker="s", color="#F58518",
             linewidth=2, label="OIS")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(probs["meeting_label"], rotation=45, ha="right")
axes[1].set_ylabel("Cumulative 25bp Cuts")
axes[1].set_title("Cumulative Implied Easing")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Calendar Spreads

In [ ]:
spreads = analyzer.calendar_spreads()

if not spreads.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(spreads))
    w = 0.35
    ax.bar(x - w/2, spreads["sofr_spread_bps"].fillna(0), w,
           color="#4C78A8", alpha=0.8, label="SOFR")
    ax.bar(x + w/2, spreads["ois_spread_bps"].fillna(0), w,
           color="#F58518", alpha=0.8, label="OIS")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(spreads["spread_label"], rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Spread (bps)")
    ax.set_title("FOMC Calendar Spreads: Meeting-to-Meeting")
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    display(spreads[["spread_label", "sofr_spread_bps", "ois_spread_bps"]].round(1))

## 7. FOMC Trade Flow — SOFR vs Fed Funds

Classify trades by rate index (UPI underlier) and analyze by meeting.

In [ ]:
fomc_trades = analyzer.fomc_trades.copy()

if not fomc_trades.empty:
    fomc_trades["rate_index"] = fomc_trades["upi_underlier_name"].apply(classify_rate_index)
    fomc_trades["venue"] = fomc_trades["platform_identifier"].apply(classify_venue)
    
    # Summary by index
    for idx in ["SOFR", "FED_FUNDS"]:
        sub = fomc_trades[fomc_trades["rate_index"] == idx]
        if sub.empty:
            continue
        d2d = (sub["venue"] == "D2D").sum()
        print(f"\n{idx}: {len(sub):,} trades, DV01 ${sub['dv01'].sum()/1e9:.0f}B, "
              f"D2D: {d2d} ({d2d/len(sub)*100:.0f}%)")
        
        # By meeting
        mtg = sub.groupby("meeting_label").agg(
            count=("dv01", "size"),
            total_dv01=("dv01", "sum"),
            total_notional=("notional", "sum"),
        ).sort_values("total_dv01", ascending=False)
        mtg["total_dv01"] = mtg["total_dv01"].apply(lambda x: f"${x/1e9:.0f}B")
        mtg["total_notional"] = mtg["total_notional"].apply(lambda x: f"${x/1e6:.0f}M")
        display(mtg.head(8))
else:
    print("No FOMC-dated trades in data.")

## 8. Trade Quality — Flag Outliers & UFRO

In [ ]:
if not fomc_trades.empty:
    flagged = flag_outliers(fomc_trades)
    
    ufro = flagged[flagged["is_ufro"] == True]
    off_market = flagged[flagged["is_off_market"] == True]
    
    print(f"Total FOMC trades: {len(flagged):,}")
    print(f"  UFRO (upfront payment): {len(ufro)} ({len(ufro)/len(flagged)*100:.1f}%)")
    print(f"  Off-market (>10bp from median): {len(off_market)} ({len(off_market)/len(flagged)*100:.1f}%)")
    
    if not ufro.empty:
        print(f"\nUFRO trades (rate != economic rate, upfront settles NPV diff):")
        ufro_display = ufro[["execution_timestamp", "meeting_label", "notional", 
                             "fixed_rate", "ufro_amount", "platform_identifier"]].copy()
        ufro_display["fixed_rate"] = ufro_display["fixed_rate"].apply(
            lambda x: f"{x*100:.3f}%" if pd.notna(x) else "N/A")
        ufro_display["notional"] = ufro_display["notional"].apply(
            lambda x: f"${x/1e6:.0f}M" if pd.notna(x) else "N/A")
        ufro_display["ufro_amount"] = ufro_display["ufro_amount"].apply(
            lambda x: f"${x:,.0f}" if pd.notna(x) else "")
        display(ufro_display.head(15))
else:
    print("No FOMC trades to flag.")

## 9. Intraday Execution Timing

In [ ]:
if not fomc_trades.empty:
    hourly = hourly_distribution(fomc_trades, timezone="America/New_York")
    
    if not hourly.empty:
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.bar(hourly.index, hourly["count"], color="#4C78A8", alpha=0.8)
        ax.set_xlabel("Hour (ET)")
        ax.set_ylabel("Trade Count")
        ax.set_title("FOMC Swap Execution Timing (Eastern Time)")
        ax.set_xticks(range(0, 24))
        ax.set_xticklabels([f"{h:02d}" for h in range(24)])
        plt.tight_layout()
        plt.show()
else:
    print("No FOMC trades for timing analysis.")

## 10. Trade Clusters — Detect Disguised Calendar Spreads

In [ ]:
if not fomc_trades.empty and "execution_timestamp" in fomc_trades.columns:
    clustered = trade_clustering(fomc_trades, gap_seconds=120)
    
    # Multi-leg clusters (2+ trades within 120s)
    cluster_sizes = clustered.groupby("cluster_id").size()
    multi_leg = cluster_sizes[cluster_sizes >= 2]
    
    print(f"Trade clusters (120s window): {len(cluster_sizes)} total")
    print(f"  Multi-leg clusters: {len(multi_leg)} ({multi_leg.sum()} trades)")
    print(f"  Single trades: {(cluster_sizes == 1).sum()}")
    
    # Show largest clusters
    if not multi_leg.empty:
        print(f"\nTop multi-leg clusters:")
        for cid in multi_leg.nlargest(10).index:
            cluster = clustered[clustered["cluster_id"] == cid].sort_values("execution_timestamp")
            meetings = sorted(cluster["meeting_label"].unique())
            mtg_str = "/".join(meetings)
            ts_first = str(cluster["execution_timestamp"].iloc[0])[:19]
            total_not = cluster["notional"].sum() / 1e6
            n_legs = len(cluster)
            multi = " ** CALENDAR **" if len(meetings) > 1 else ""
            print(f"  {ts_first}  {n_legs} legs  [{mtg_str}]  ${total_not:.0f}M{multi}")
else:
    print("No FOMC trades for clustering.")

## 11. Summary Dashboard

In [ ]:
summary = analyzer.summary()

print("=" * 70)
print("FOMC ANALYTICS SUMMARY")
print("=" * 70)
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print("=" * 70)